In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib, json, shutil, zipfile, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from scipy.stats import spearmanr
try:
    from IPython.display import display, HTML
except ImportError:
    display = print
    HTML = lambda text: text


ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/hitting'
OUT = ROOT / 'results/stride_movement_consistency'
FIG = ROOT / 'figures/stride_movement_consistency'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True
LOOKBACK_S = .150
FILTER_HZ = None
FILTER_PADDING_S = .250
MIN_VALID_SWINGS = 5
REVERSAL_THRESHOLD_DEG = .5
BOOTSTRAPS = 1000
SEED = 20261016
SELECTED_HITTER_SESSION = None
PROGRESS = np.linspace(0, 1, 101)
EVENTS = ['fp_10_time', 'fp_100_time', 'contact_time']
SEGMENTS = ('pelvis', 'torso')
ANGLE_COLS = [s + '_angle_x' for s in SEGMENTS]
OUTCOMES = {'bat_speed_mph': 'bat_speed_mph_contact_x',
            'exit_velocity_mph': 'exit_velo_mph_x'}
ARCHIVE_HASHES = {
    'hitting_joint_angles.zip': '98a405c5c4a27c1a599d3e30b39a3087180cc4d4bb69faf62204a1c06881be95'}
BASE = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
CSV_BASE = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_hitting/data/'
COLORS = {'pelvis': '#167D9A', 'torso': '#A83C51'}
plt.style.use('seaborn-v0_8-whitegrid')
print('Working directory:', ROOT)
print('Descriptive study: no ideal posture, stability grade, or causal benefit is assumed.')
print('Approach: 150 ms before foot contact; landing: foot contact to full plant.')
print('Phase eligibility is independent. Ball-contact annotation is not required for either phase.')
print('Reversal frequency is the percentage of valid session swings with at least one reversal.')
print('Rank agreement is a sensitivity check, not a reliability estimate or a performance grade.')


In [ ]:
def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, target):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {target}. Supply official data or set AUTO_DOWNLOAD=True.')
    print('Downloading', target.name, flush=True)
    temporary = target.with_suffix(target.suffix + '.part')
    urlretrieve(url, temporary)
    temporary.replace(target)

def ensure_archive(archive_name, csv_name):
    target = RAW / csv_name
    if target.exists():
        return target
    archive = RAW / archive_name
    if not archive.exists():
        download(BASE + archive_name, archive)
    if file_hash(archive) != ARCHIVE_HASHES[archive_name]:
        raise ValueError(f'Checksum mismatch: {archive.name}; remove the archive and retry.')
    with zipfile.ZipFile(archive) as z:
        members = [name for name in z.namelist() if Path(name).name == csv_name]
        if len(members) != 1:
            raise ValueError(f'Expected exactly one {csv_name} in {archive_name}')
        temporary = target.with_suffix('.csv.part')
        with z.open(members[0]) as source, temporary.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        temporary.replace(target)
    return target

ensure_archive('hitting_joint_angles.zip', 'joint_angles.csv')
if not (RAW / 'metadata.csv').exists():
    download(CSV_BASE + 'metadata.csv', RAW / 'metadata.csv')
angles = pd.read_csv(RAW / 'joint_angles.csv',
                     usecols=['session_swing', 'time'] + EVENTS + ANGLE_COLS,
                     dtype={'session_swing': str})
metadata = pd.read_csv(RAW / 'metadata.csv',
                       dtype={'session_swing': str, 'user': str, 'session': str})
required_metadata = ['session_swing', 'user', 'session', 'hitter_side']
if not set(required_metadata).issubset(metadata.columns):
    raise ValueError('Missing required metadata columns')
if metadata.session_swing.isna().any() or metadata.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate metadata swing IDs')
if angles.session_swing.isna().any() or angles.duplicated(['session_swing', 'time']).any():
    raise ValueError('Missing swing IDs or duplicate angle swing/time keys')
meta = metadata.set_index('session_swing')
groups = {str(k): g.sort_values('time').reset_index(drop=True)
          for k, g in angles.groupby('session_swing', sort=False)}

poi = pd.DataFrame(columns=['session_swing'] + list(OUTCOMES.values()))
try:
    if not (RAW / 'poi_metrics.csv').exists():
        download(CSV_BASE + 'poi/poi_metrics.csv', RAW / 'poi_metrics.csv')
    poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
except (OSError, FileNotFoundError) as exc:
    print('Optional outcomes unavailable; descriptive analysis continues:', exc)
if poi.session_swing.isna().any() or poi.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate POI swing IDs')
for column in OUTCOMES.values():
    if column not in poi:
        poi[column] = np.nan
print(f'{len(groups):,} angle trials; {metadata.user.nunique():,} hitters in metadata')

In [ ]:
def event_values(g, required):
    events = {}
    for name in required:
        values = g[name].to_numpy(float)
        if not np.isfinite(values).all() or np.ptp(values) > 1e-7:
            raise ValueError('missing or inconsistent '+name)
        events[name] = float(values[0])
    if 'fp_100_time' in required and events['fp_100_time'] <= events['fp_10_time']:
        raise ValueError('unordered foot contact and plant')
    return events

def meaningful_reversals(a, threshold=REVERSAL_THRESHOLD_DEG):
    a = np.asarray(a, float)
    if threshold <= 0 or len(a) < 2 or not np.isfinite(a).all():
        raise ValueError('invalid reversal inputs')
    low = high = a[0]
    direction = 0
    count = 0
    for value in a[1:]:
        if direction == 0:
            low, high = min(low, value), max(high, value)
            if value - low >= threshold:
                direction, high = 1, value
            elif high - value >= threshold:
                direction, low = -1, value
        elif direction == 1:
            high = max(high, value)
            if high - value >= threshold:
                count += 1
                direction, low = -1, value
        else:
            low = min(low, value)
            if value - low >= threshold:
                count += 1
                direction, high = 1, value
    return count

def path_metrics(t, a, threshold=REVERSAL_THRESHOLD_DEG):
    t, a = np.asarray(t, float), np.asarray(a, float)
    if len(t) < 3 or len(t) != len(a) or not np.isfinite(t).all() or not np.isfinite(a).all():
        raise ValueError('invalid metric arrays')
    h = np.diff(t)
    if np.any(h <= 0):
        raise ValueError('nonincreasing metric times')
    duration = t[-1] - t[0]
    b = a - a[0]
    mean = np.sum(h * (b[:-1] + b[1:]) / 2) / duration
    second = np.sum(h * (b[:-1]**2 + b[:-1]*b[1:] + b[1:]**2) / 3) / duration
    net = float(a[-1] - a[0])
    travel = float(np.abs(np.diff(a)).sum())
    return {'angle_sd_deg': float(np.sqrt(max(0., second - mean**2))),
            'net_change_deg': net, 'total_travel_deg': travel,
            'excess_travel_deg': max(0., travel - abs(net)),
            'reversals': meaningful_reversals(a, threshold),
            'range_deg': float(np.ptp(a)), 'start_deg': float(a[0]), 'end_deg': float(a[-1])}

def extract_phase(g, phase, lookback=LOOKBACK_S, cutoff=FILTER_HZ,
                  reversal=REVERSAL_THRESHOLD_DEG):
    required = ['fp_10_time'] if phase == 'approach' else ['fp_10_time','fp_100_time']
    events = event_values(g, required)
    end = events['fp_10_time'] if phase == 'approach' else events['fp_100_time']
    start = end-lookback if phase == 'approach' else events['fp_10_time']
    full_t = g.time.to_numpy(float)
    if not np.isfinite(full_t).all() or np.any(np.diff(full_t) <= 0):
        raise ValueError('invalid time grid')
    pad = FILTER_PADDING_S if cutoff is not None else 0.
    if full_t[0] > start-pad+1e-10 or full_t[-1] < end+pad-1e-10:
        raise ValueError('insufficient window/filter context coverage')
    left = max(0,np.searchsorted(full_t,start-pad,side='right')-1)
    right = min(len(full_t),np.searchsorted(full_t,end+pad,side='left')+1)
    t = full_t[left:right]
    if len(t)<8:
        raise ValueError('too few phase/context samples')
    dt = float(np.median(np.diff(t)))
    if not np.allclose(np.diff(t),dt,rtol=.03,atol=.00011):
        raise ValueError('sampling gap or irregular grid')
    if end-start < 7*dt-1e-9:
        raise ValueError('phase shorter than seven sampling intervals')
    row = {phase+'_duration_ms':1000*(end-start),phase+'_sample_interval_ms':1000*dt}
    curves = {}
    tt = np.r_[start,t[(t>start+1e-10)&(t<end-1e-10)],end]
    for segment in SEGMENTS:
        a = g[segment+'_angle_x'].to_numpy(float)[left:right]
        if not np.isfinite(a).all():
            raise ValueError('nonfinite angle in requested window/context')
        a = np.rad2deg(np.unwrap(np.deg2rad(a)))
        if cutoff is not None:
            if not 0 < cutoff < .5/dt:
                raise ValueError('invalid filter cutoff')
            a = sosfiltfilt(butter(4,cutoff,fs=1/dt,output='sos'),a)
        aa = np.interp(tt,t,a)
        row.update({f'{phase}_{segment}_{name}':value for name,value in path_metrics(tt,aa,reversal).items()})
        curves[(phase,segment)] = np.interp(start+PROGRESS*(end-start),tt,aa)
    return row,curves

def extract_trial(g,lookback=LOOKBACK_S,cutoff=FILTER_HZ,reversal=REVERSAL_THRESHOLD_DEG):
    row,curves = {},{}
    for phase in ('approach','landing'):
        try:
            measurements,trace = extract_phase(g,phase,lookback,cutoff,reversal)
            row.update(measurements)
            curves.update(trace)
            status = 'retained'
        except (ValueError,KeyError,TypeError) as exc:
            status = str(exc)
        row[phase+'_valid'] = status == 'retained'
        row[phase+'_status'] = status
    return row,curves

def build_table(lookback=LOOKBACK_S,cutoff=FILTER_HZ,reversal=REVERSAL_THRESHOLD_DEG):
    rows,audit,curves = [],[],{}
    for key,g in groups.items():
        try:
            if key not in meta.index:
                raise ValueError('missing metadata match')
            identity = meta.loc[key]
            if identity[['user','session']].isna().any() or identity.hitter_side not in ('L','R'):
                raise ValueError('missing identity or handedness')
            row,trace = extract_trial(g,lookback,cutoff,reversal)
            retained = row['approach_valid'] or row['landing_valid']
            if retained:
                rows.append({'session_swing':key,'hitter_id':str(identity.user),
                             'session_id':str(identity.session),'hitter_side':identity.hitter_side,**row})
                curves[key] = trace
            audit.append({'session_swing':key,'status':'retained_in_at_least_one_phase' if retained else 'excluded_both_phases',
                          'approach_status':row['approach_status'],'landing_status':row['landing_status']})
        except (ValueError,KeyError,TypeError) as exc:
            audit.append({'session_swing':key,'status':str(exc),
                          'approach_status':str(exc),'landing_status':str(exc)})
    return pd.DataFrame(rows),pd.DataFrame(audit),curves

def trajectory_variability(matrix, relative=False):
    m = np.asarray(matrix,float)
    if m.ndim != 2 or len(m)<2 or not np.isfinite(m).all():
        raise ValueError('invalid trajectory matrix')
    if relative:
        m = m-m[:,[0]]
    return float(np.sqrt(np.mean(np.var(m,axis=0,ddof=1))))

def session_profiles(features,curves,minimum=MIN_VALID_SWINGS):
    rows = []
    for (hitter,session),g in features.groupby(['hitter_id','session_id']):
        row = {'hitter_id':hitter,'session_id':session,'swings':len(g)}
        eligible = False
        for phase in ('approach','landing'):
            valid = g.loc[g[phase+'_valid']].copy()
            count = len(valid)
            row[phase+'_swings'] = count
            row[phase+'_eligible'] = count >= minimum
            metrics = [c for c in features if c.startswith((phase+'_pelvis_',phase+'_torso_'))]
            for c in metrics:
                row[c] = float(valid[c].median()) if count>=minimum else np.nan
            row[phase+'_duration_ms'] = valid[phase+'_duration_ms'].median() if count>=minimum else np.nan
            for segment in SEGMENTS:
                prefix = f'{phase}_{segment}_'
                for metric in ('absolute_trajectory_sd_deg','relative_trajectory_sd_deg','reversal_frequency_pct'):
                    row[prefix+metric] = np.nan
                if count>=minimum:
                    matrix = np.stack([curves[k][(phase,segment)] for k in valid.session_swing])
                    row[prefix+'absolute_trajectory_sd_deg'] = trajectory_variability(matrix)
                    row[prefix+'relative_trajectory_sd_deg'] = trajectory_variability(matrix,True)
                    row[prefix+'reversal_frequency_pct'] = 100*valid[prefix+'reversals'].gt(0).mean()
            eligible |= count>=minimum
        for outcome in OUTCOMES:
            if outcome in g:
                v = g.loc[g.approach_valid,outcome].dropna()
                row[outcome+'_n'] = len(v)
                row[outcome+'_mean'] = v.mean() if len(v)>=minimum else np.nan
                row[outcome+'_sd'] = v.std(ddof=1) if len(v)>=minimum else np.nan
        if eligible:
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def run_measurement_checks():
    t=np.linspace(0,.15,55)
    a=10*t/.15
    p=path_metrics(t,a)
    assert np.isclose(p['angle_sd_deg'],10/np.sqrt(12))
    assert np.isclose(p['excess_travel_deg'],0) and p['reversals']==0
    assert np.isclose(path_metrics(t,a+80)['angle_sd_deg'],p['angle_sd_deg'])
    q=path_metrics(np.array([0.,.075,.15]),np.array([0.,5.,0.]))
    assert np.isclose(q['net_change_deg'],0) and np.isclose(q['excess_travel_deg'],10)
    assert q['reversals']==1
    assert meaningful_reversals([0,.1,-.1,.1,0],.5)==0
    assert meaningful_reversals([0,2,0,2,0],.5)==3
    m=np.stack([a,a+2])
    assert trajectory_variability(m)>0 and np.isclose(trajectory_variability(m,True),0)
    assert np.isclose(trajectory_variability(np.stack([a,a]),True),0)
    times=np.arange(-.6,.61,1/360)
    trial=pd.DataFrame({'time':times,'fp_10_time':0.,'fp_100_time':.06,
                        'contact_time':.19,'pelvis_angle_x':times*30,
                        'torso_angle_x':times*40})
    r,c=extract_trial(trial)
    assert np.isclose(r['approach_pelvis_net_change_deg'],4.5)
    assert np.isclose(r['landing_torso_net_change_deg'],2.4)
    wrapped=trial.copy()
    wrapped['pelvis_angle_x']=((179+times*30+180)%360)-180
    rw,_=extract_trial(wrapped)
    assert np.isclose(rw['approach_pelvis_net_change_deg'],4.5)
    filtered,_=extract_trial(trial,cutoff=12.)
    assert np.isfinite(filtered['approach_pelvis_angle_sd_deg'])
    for bad in [trial.loc[~trial.index.isin([np.abs(times+.05).argmin()])],
                trial.assign(pelvis_angle_x=np.where(np.abs(times+.05)<.002,np.nan,times*30))]:
        try:
            extract_phase(bad,'approach')
        except ValueError:
            pass
        else:
            raise AssertionError('Invalid trial was accepted')
    for invalid_plant in [-.01,np.nan,.01]:
        phase_row,trace=extract_trial(trial.assign(fp_100_time=invalid_plant))
        assert phase_row['approach_valid'] and not phase_row['landing_valid']
        assert ('approach','pelvis') in trace and ('landing','pelvis') not in trace
    phase_row,trace=extract_trial(trial.assign(contact_time=np.nan))
    assert phase_row['approach_valid'] and phase_row['landing_valid']
    phase_row,trace=extract_trial(trial.assign(pelvis_angle_x=np.where(times>.03,np.nan,times*30)))
    assert phase_row['approach_valid'] and not phase_row['landing_valid']
    temporary=[]
    temporary_curves={}
    for j in range(5):
        phase_row,trace=extract_trial(trial)
        phase_row.update(session_swing=str(j),hitter_id='test',session_id='test')
        phase_row['approach_pelvis_reversals']=int(j==0)
        temporary.append(phase_row)
        temporary_curves[str(j)]=trace
    test_profile=session_profiles(pd.DataFrame(temporary),temporary_curves).iloc[0]
    assert test_profile['approach_pelvis_reversals']==0
    assert np.isclose(test_profile['approach_pelvis_reversal_frequency_pct'],20)
    print('Independent phase eligibility and reversal-frequency checks passed.')
    print('Measurement checks passed: exact boundaries, SD, recovered travel, reversal threshold,')
    print('trajectory offsets, unwrapping, filtering, gaps, nonfinite values and event ordering.')
run_measurement_checks()
features,audit,curves=build_table()
audit.to_csv(OUT/'exclusion_audit.csv',index=False)
if features.empty:
    display(audit.status.value_counts())
    raise ValueError('No eligible swings; inspect the exclusion audit and input conventions.')
for outcome,column in OUTCOMES.items():
    features=features.merge(poi[['session_swing',column]].rename(columns={column:outcome}),
                            on='session_swing',how='left',validate='one_to_one')
    features[outcome]=pd.to_numeric(features[outcome],errors='coerce').replace([np.inf,-np.inf],np.nan)
features.to_csv(OUT/'swing_metrics.csv',index=False)
profiles=session_profiles(features,curves)
if profiles.empty:
    raise ValueError('No sessions meet MIN_VALID_SWINGS; inspect counts before lowering the minimum.')
profiles.to_csv(OUT/'session_profiles.csv',index=False)
print(f'{len(features):,}/{len(groups):,} swings retained in at least one phase.')
cohort_rows=[]
for phase in ('approach','landing'):
    valid=features.loc[features[phase+'_valid']]
    eligible=profiles.loc[profiles[phase+'_eligible']]
    cohort_rows.append({'phase':phase,'swings':len(valid),'hitters':valid.hitter_id.nunique(),
                        'eligible_sessions':len(eligible),'eligible_hitters':eligible.hitter_id.nunique()})
    print(phase.title()+' exclusion audit:')
    display(audit[phase+'_status'].value_counts().rename_axis('status').to_frame('swings'))
cohorts=pd.DataFrame(cohort_rows)
cohorts.to_csv(OUT/'phase_cohorts.csv',index=False)
display(cohorts)
approach_profiles=profiles.loc[profiles.approach_eligible].copy()
display(profiles[['hitter_id','session_id','swings','approach_pelvis_angle_sd_deg',
                  'approach_pelvis_relative_trajectory_sd_deg','approach_torso_relative_trajectory_sd_deg']].head(12))

In [ ]:
def save_figure(fig,name):
    fig.tight_layout()
    fig.savefig(FIG/(name+'.png'),dpi=160,bbox_inches='tight')
    plt.show()

net='approach_pelvis_net_change_deg'
excess='approach_pelvis_excess_travel_deg'
approach_features=features.loc[features.approach_valid].copy()
q1,q3=approach_features[net].quantile([.25,.75])
candidates=approach_features.loc[approach_features[net].between(q1,q3)].sort_values(net)
best=None
for i in range(len(candidates)):
    same=candidates.iloc[i+1:]
    same=same.loc[(same[net]-candidates.iloc[i][net]).abs()<=.5]
    if len(same):
        j=(same[excess]-candidates.iloc[i][excess]).abs().idxmax()
        gap=abs(same.loc[j,excess]-candidates.iloc[i][excess])
        if best is None or gap>best[0]:
            best=(gap,candidates.iloc[i],same.loc[j])
if best is None:
    mid=approach_features.sort_values(net).iloc[max(0,len(approach_features)//2-1):len(approach_features)//2+1]
    example_rows=[row for _,row in mid.iterrows()]
else:
    example_rows=[best[1],best[2]]
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for ax,s in zip(axes,SEGMENTS):
    for row in example_rows:
        a=curves[row.session_swing][('approach',s)]
        ax.plot((PROGRESS-1)*LOOKBACK_S*1000,a-a[0],lw=2,
                label=f"{row.session_swing}: net {row[f'approach_{s}_net_change_deg']:+.2f}°, excess {row[f'approach_{s}_excess_travel_deg']:.2f}°")
    ax.set(xlabel='Milliseconds before front-foot contact',ylabel='Angle change from start (°)',title=s.title())
    ax.legend(fontsize=8)
fig.suptitle('Similar endpoints can conceal different movement paths')
save_figure(fig,'matched_endpoint_examples')
pd.DataFrame(example_rows).to_csv(OUT/'selected_example_swings.csv',index=False)

ranked=approach_profiles.sort_values('approach_pelvis_relative_trajectory_sd_deg')
if SELECTED_HITTER_SESSION is None:
    selected=ranked.iloc[sorted(set([0,len(ranked)//2,len(ranked)-1]))]
else:
    h,se=map(str,SELECTED_HITTER_SESSION)
    selected=approach_profiles.loc[approach_profiles.hitter_id.eq(h)&approach_profiles.session_id.eq(se)]
    if selected.empty:
        raise ValueError('SELECTED_HITTER_SESSION is not an eligible profile')
fig,axes=plt.subplots(len(selected),2,figsize=(13,3.8*len(selected)),squeeze=False)
for axs,(_,profile) in zip(axes,selected.iterrows()):
    sub=features.loc[features.approach_valid&features.hitter_id.eq(profile.hitter_id)&features.session_id.eq(profile.session_id)]
    for ax,s in zip(axs,SEGMENTS):
        m=np.stack([curves[k][('approach',s)] for k in sub.session_swing])
        m=m-m[:,[0]]
        x=(PROGRESS-1)*LOOKBACK_S*1000
        for a in m:
            ax.plot(x,a,color=COLORS[s],alpha=.3,lw=1)
        mean,sd=m.mean(axis=0),m.std(axis=0,ddof=1)
        ax.fill_between(x,mean-sd,mean+sd,color=COLORS[s],alpha=.14,label='Across-swing ±1 SD')
        ax.plot(x,mean,color=COLORS[s],lw=2.5,label='Mean trajectory')
        v=profile[f'approach_{s}_relative_trajectory_sd_deg']
        ax.set(title=f'{s.title()} · hitter {profile.hitter_id}, session {profile.session_id} · n={len(m)}, trajectory SD={v:.2f}°',
               xlabel='Milliseconds before front-foot contact',ylabel='Angle change from start (°)')
        ax.legend(fontsize=8)
save_figure(fig,'session_trajectory_examples')

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,s in zip(axes,SEGMENTS):
    ax.scatter(approach_profiles[f'approach_{s}_angle_sd_deg'],approach_profiles[f'approach_{s}_relative_trajectory_sd_deg'],
               color=COLORS[s],alpha=.7,s=35)
    ax.set(xlabel='Median within-swing angle SD (°)',ylabel='Across-swing trajectory SD (°)',
           title=s.title()+' · each point is a hitter-session')
save_figure(fig,'movement_amount_vs_repeatability')

landing_profiles=profiles.loc[profiles.landing_eligible].sort_values('landing_pelvis_relative_trajectory_sd_deg')
if landing_profiles.empty:
    raise ValueError('No eligible landing sessions for visualization')
profile=landing_profiles.iloc[len(landing_profiles)//2]
sub=features.loc[features.landing_valid&features.hitter_id.eq(profile.hitter_id)&features.session_id.eq(profile.session_id)]
fig,axes=plt.subplots(2,2,figsize=(13,8))
for axs,s in zip(axes,SEGMENTS):
    m=np.stack([curves[k][('landing',s)] for k in sub.session_swing])
    for ax,relative in zip(axs,[False,True]):
        view=m-m[:,[0]] if relative else m
        for a in view:
            ax.plot(PROGRESS*100,a,color=COLORS[s],alpha=.3,lw=1)
        ax.plot(PROGRESS*100,view.mean(axis=0),color=COLORS[s],lw=2.5)
        ax.set(xlabel='Landing progress: foot contact to full plant (%)',
               ylabel='Change from contact (°)' if relative else 'Published angle (°)',
               title=s.title()+(' · start-relative' if relative else ' · absolute posture'))
fig.suptitle(f'Landing repeatability · hitter {profile.hitter_id}, session {profile.session_id} · n={len(sub)}')
save_figure(fig,'landing_absolute_and_relative_trajectories')

In [ ]:
def mean_bootstrap(values):
    values=np.asarray(values,float)
    values=values[np.isfinite(values)]
    if not len(values):
        return np.nan,np.nan,np.nan
    estimate=float(values.mean())
    if len(values)<5:
        return estimate,np.nan,np.nan
    rng=np.random.default_rng(SEED)
    draws=values[rng.integers(0,len(values),size=(BOOTSTRAPS,len(values)))].mean(axis=1)
    lo,hi=np.quantile(draws,[.025,.975])
    return estimate,float(lo),float(hi)

profile_metrics=[c for c in profiles if c.startswith(('approach_pelvis_','approach_torso_',
                                                      'landing_pelvis_','landing_torso_'))]
hitters=profiles.groupby('hitter_id')[profile_metrics].mean()
summary=[]
for metric in profile_metrics:
    estimate,lo,hi=mean_bootstrap(hitters[metric])
    summary.append({'metric':metric,'hitters':int(hitters[metric].notna().sum()),'mean':estimate,'ci_low':lo,'ci_high':hi})
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'hitter_weighted_summary.csv',index=False)
display(summary.loc[summary.metric.str.contains('trajectory|excess_travel|reversals|reversal_frequency')])

def safe_rho(x,y):
    x,y=np.asarray(x,float),np.asarray(y,float)
    good=np.isfinite(x)&np.isfinite(y)
    x,y=x[good],y[good]
    return float(spearmanr(x,y).statistic) if len(x)>=3 and np.ptp(x)>0 and np.ptp(y)>0 else np.nan

settings={'primary':(LOOKBACK_S,None,REVERSAL_THRESHOLD_DEG),
          'window_100ms':(.100,None,REVERSAL_THRESHOLD_DEG),
          'window_200ms':(.200,None,REVERSAL_THRESHOLD_DEG),
          'filter_8Hz':(LOOKBACK_S,8.,REVERSAL_THRESHOLD_DEG),
          'filter_12Hz':(LOOKBACK_S,12.,REVERSAL_THRESHOLD_DEG),
          'filter_16Hz':(LOOKBACK_S,16.,REVERSAL_THRESHOLD_DEG),
          'reversal_025deg':(LOOKBACK_S,None,.25),
          'reversal_100deg':(LOOKBACK_S,None,1.)}
configuration_tables={'primary':features}
configuration_curves={'primary':curves}
for label,(window,cutoff,threshold) in settings.items():
    if label=='primary':
        continue
    table,exclusions,trace=build_table(window,cutoff,threshold)
    exclusions.to_csv(OUT/(label+'_exclusion_audit.csv'),index=False)
    configuration_tables[label]=table
    configuration_curves[label]=trace
    table.to_csv(OUT/(label+'_swing_metrics.csv'),index=False)
    print(label,':',len(table),'retained swings',flush=True)

sensitivity=[]
for family,labels in [('window',['primary','window_100ms','window_200ms']),
                      ('filter',['primary','filter_8Hz','filter_12Hz','filter_16Hz']),
                      ('reversal',['primary','reversal_025deg','reversal_100deg'])]:
    for phase in ('approach','landing'):
        common=set(features.loc[features[phase+'_valid'],'session_swing'])
        for label in labels:
            table=configuration_tables[label]
            common &= set(table.loc[table[phase+'_valid'],'session_swing']) if len(table) else set()
        base=None
        for label in labels:
            table=configuration_tables[label]
            if not common:
                sensitivity.append({'family':family,'phase':phase,'setting':label,'common_swings':0,'sessions':0})
                continue
            sub=table.loc[table.session_swing.isin(common)]
            p=session_profiles(sub,configuration_curves[label])
            if len(p):
                p=p.loc[p[phase+'_eligible']].copy()
            p.to_csv(OUT/(family+'_'+phase+'_'+label+'_common_session_profiles.csv'),index=False)
            if p.empty:
                sensitivity.append({'family':family,'phase':phase,'setting':label,'common_swings':len(common),'sessions':0})
                continue
            if label=='primary':
                base=p
            for segment in SEGMENTS:
                for suffix in ['angle_sd_deg','excess_travel_deg','reversals',
                               'reversal_frequency_pct','relative_trajectory_sd_deg']:
                    metric=f'{phase}_{segment}_{suffix}'
                    paired=base[['hitter_id','session_id',metric]].merge(
                        p[['hitter_id','session_id',metric]],on=['hitter_id','session_id'],
                        suffixes=('_base','_alt'),validate='one_to_one')
                    difference=(paired[metric+'_alt']-paired[metric+'_base']).abs()
                    hv=p.groupby('hitter_id')[metric].mean()
                    value,lo,hi=mean_bootstrap(hv)
                    sensitivity.append({'family':family,'phase':phase,'segment':segment,
                        'setting':label,'common_swings':len(common),'sessions':len(p),
                        'hitters':p.hitter_id.nunique(),'metric':metric,'measure':suffix,
                        'hitter_weighted_mean':value,'ci_low':lo,'ci_high':hi,
                        'session_rank_rho':safe_rho(paired[metric+'_base'],paired[metric+'_alt']),
                        'median_absolute_session_change':difference.median()})
sensitivity=pd.DataFrame(sensitivity)
sensitivity.to_csv(OUT/'setting_sensitivity.csv',index=False)
compact_parts=[]
for phase in ('approach','landing'):
    for segment in SEGMENTS:
        sub=sensitivity.loc[sensitivity.phase.eq(phase)&sensitivity.segment.eq(segment)]
        index=['family','setting','common_swings','hitters']
        means=sub.pivot(index=index,columns='measure',values='hitter_weighted_mean')
        ranks=sub.pivot(index=index,columns='measure',values='session_rank_rho')
        compact=means[['relative_trajectory_sd_deg','excess_travel_deg','reversal_frequency_pct']].rename(
            columns={'relative_trajectory_sd_deg':'trajectory_SD_deg',
                     'excess_travel_deg':'excess_deg','reversal_frequency_pct':'reversal_pct'})
        for original,label in [('relative_trajectory_sd_deg','trajectory_rank_rho'),
                               ('excess_travel_deg','excess_rank_rho'),
                               ('reversal_frequency_pct','reversal_rank_rho')]:
            compact[label]=ranks[original]
        compact=compact.reset_index()
        compact.insert(0,'segment',segment)
        compact.insert(0,'phase',phase)
        compact_parts.append(compact)
        print(phase.title()+' / '+segment.title()+' — paired common-cohort sensitivity')
        with pd.option_context('display.max_rows',None,'display.max_columns',None,
                               'display.width',180,'display.precision',3):
            display(compact.drop(columns=['phase','segment']))
compact_sensitivity=pd.concat(compact_parts,ignore_index=True)
compact_sensitivity.to_csv(OUT/'compact_sensitivity.csv',index=False)
plot_data=sensitivity.loc[sensitivity.family.isin(['window','filter'])&
                         sensitivity.measure.eq('relative_trajectory_sd_deg')&
                         sensitivity.setting.ne('primary')]
fig,axes=plt.subplots(1,2,figsize=(13,5),squeeze=False)
for ax,phase in zip(axes[0],('approach','landing')):
    panel=plot_data.loc[plot_data.phase.eq(phase)].pivot(index='setting',columns='segment',values='session_rank_rho')
    im=ax.imshow(panel.to_numpy(),vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
    ax.grid(False)
    ax.set_xticks(range(len(panel.columns)),panel.columns)
    ax.set_yticks(range(len(panel)),panel.index)
    for i in range(len(panel)):
        for j in range(len(panel.columns)):
            value=panel.iloc[i,j]
            ax.text(j,i,f'{value:.3f}',ha='center',va='center',color='white' if value>.65 else 'black')
    ax.set_title(phase.title()+' trajectory-variability rank agreement')
    fig.colorbar(im,ax=ax,label='Spearman correlation with primary')
save_figure(fig,'trajectory_rank_sensitivity')
minimum_rows=[]
for minimum in [4,5,6]:
    p=session_profiles(features,curves,minimum)
    for phase in ('approach','landing'):
        phase_profiles=p.loc[p[phase+'_eligible']] if len(p) else pd.DataFrame()
        for segment in SEGMENTS:
            metric=f'{phase}_{segment}_relative_trajectory_sd_deg'
            hv=phase_profiles.groupby('hitter_id')[metric].mean() if len(phase_profiles) else []
            mean,lo,hi=mean_bootstrap(hv)
            minimum_rows.append({'phase':phase,'minimum_swings':minimum,'segment':segment,
                'sessions':len(phase_profiles),'hitters':phase_profiles.hitter_id.nunique() if len(phase_profiles) else 0,
                'mean_relative_trajectory_sd_deg':mean,'ci_low':lo,'ci_high':hi})
minimum_sensitivity=pd.DataFrame(minimum_rows)
minimum_sensitivity.to_csv(OUT/'minimum_swings_sensitivity.csv',index=False)
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(minimum_sensitivity)

In [ ]:
def association(data,x,y,scope):
    d=data[['hitter_id',x,y]].replace([np.inf,-np.inf],np.nan).dropna()
    ids=d.hitter_id.unique()
    estimate=safe_rho(d[x],d[y]) if len(ids)>=5 else np.nan
    draws=[]
    if len(ids)>=5 and np.isfinite(estimate):
        chunks=[g[[x,y]].to_numpy(float) for _,g in d.groupby('hitter_id')]
        rng=np.random.default_rng(SEED)
        for _ in range(BOOTSTRAPS):
            b=np.concatenate([chunks[i] for i in rng.integers(0,len(chunks),len(chunks))])
            rho=safe_rho(b[:,0],b[:,1])
            if np.isfinite(rho):
                draws.append(rho)
    lo,hi=np.quantile(draws,[.025,.975]) if draws else (np.nan,np.nan)
    return {'scope':scope,'metric':x,'outcome':y,'rows':len(d),'hitters':len(ids),
            'rho':estimate,'ci_low':lo,'ci_high':hi,'valid_bootstrap_draws':len(draws)}

relationship_rows=[]
movement_metrics=[f'approach_{s}_{suffix}' for s in SEGMENTS for suffix in
                  ['angle_sd_deg','excess_travel_deg','reversals']]
repeat_metrics=[f'approach_{s}_{suffix}_trajectory_sd_deg' for s in SEGMENTS
                for suffix in ['absolute','relative']]
for outcome in OUTCOMES:
    for target in [outcome+'_mean',outcome+'_sd']:
        for metric in movement_metrics+repeat_metrics:
            complete=profiles[['hitter_id',metric,target]].dropna()
            between=complete.groupby('hitter_id')[[metric,target]].mean().reset_index()
            relationship_rows.append(association(between,metric,target,'between_hitter'))
    for metric in movement_metrics:
        d=features.loc[features.approach_valid,['hitter_id','session_id',metric,outcome]].dropna().copy()
        n=d.groupby(['hitter_id','session_id'])[metric].transform('size')
        d=d.loc[n>=MIN_VALID_SWINGS].copy()
        for column in [metric,outcome]:
            d[column]=d[column]-d.groupby(['hitter_id','session_id'])[column].transform('mean')
        relationship_rows.append(association(d,metric,outcome,'within_hitter_session_centered'))
relationships=pd.DataFrame(relationship_rows)
relationships.to_csv(OUT/'exploratory_relationships.csv',index=False)
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(relationships)

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,s in zip(axes,SEGMENTS):
    metric=f'approach_{s}_relative_trajectory_sd_deg'
    d=profiles[['hitter_id',metric,'exit_velocity_mph_sd']].dropna()
    d=d.groupby('hitter_id')[[metric,'exit_velocity_mph_sd']].mean()
    ax.scatter(d[metric],d.exit_velocity_mph_sd,color=COLORS[s],alpha=.7)
    ax.set(xlabel='Start-relative trajectory SD (°)',ylabel='Within-session exit velocity SD (mph)',
           title=s.title()+f' · {len(d)} hitters · exploratory')
save_figure(fig,'repeatability_vs_exit_velocity_variability')

In [ ]:
FOCUS_HITTER = '000078'
FOCUS_SESSION = '000268'
AUDIT_OUT = OUT / 'focused_outlier_check'
AUDIT_FIG = FIG / 'focused_outlier_check'
for folder in (AUDIT_OUT,AUDIT_FIG):
    folder.mkdir(parents=True,exist_ok=True)
focus = features.loc[features.hitter_id.eq(FOCUS_HITTER)&features.session_id.eq(FOCUS_SESSION)].copy()
if focus.empty:
    raise ValueError('The selected hitter-session has no retained trials')

def annotation_value(g,name):
    values = g[name].to_numpy(float)
    valid = bool(np.isfinite(values).all() and np.ptp(values)<=1e-7)
    return (float(values[0]) if valid else np.nan),valid

event_rows=[]
for _,trial in features.iterrows():
    g=groups[trial.session_swing]
    annotations={}
    for name in EVENTS:
        value,valid=annotation_value(g,name)
        annotations[name]=value
        annotations[name+'_consistent']=valid
    fc,plant,contact=(annotations[name] for name in EVENTS)
    event_rows.append({'session_swing':trial.session_swing,'hitter_id':trial.hitter_id,
        'session_id':trial.session_id,'approach_valid':trial.approach_valid,'landing_valid':trial.landing_valid,
        **annotations,'foot_contact_minus_ball_contact_ms':1000*(fc-contact),
        'plant_minus_ball_contact_ms':1000*(plant-contact),
        'approach_contains_ball_contact':bool(np.isfinite(fc) and np.isfinite(contact) and fc-LOOKBACK_S<=contact<=fc),
        'foot_contact_after_ball_contact':bool(np.isfinite(fc) and np.isfinite(contact) and fc>contact),
        'plant_after_ball_contact':bool(np.isfinite(plant) and np.isfinite(contact) and plant>contact)})
event_audit=pd.DataFrame(event_rows)
event_audit.to_csv(AUDIT_OUT/'retained_trial_event_context.csv',index=False)
focus_audit=focus.merge(event_audit.drop(columns=['hitter_id','session_id','approach_valid','landing_valid']),
                        on='session_swing',validate='one_to_one')
focus_audit.to_csv(AUDIT_OUT/'focus_trial_audit.csv',index=False)
print('Focused audit: hitter',FOCUS_HITTER,'session',FOCUS_SESSION)
print('Official event labels are inspected as supplied; no event is corrected or trial automatically excluded.')
print('Across retained approach trials, annotated foot contact follows ball contact in',
      int((event_audit.approach_valid & event_audit.foot_contact_after_ball_contact).sum()),'trials.')
columns=['session_swing','fp_10_time','fp_100_time','contact_time',
         'foot_contact_minus_ball_contact_ms','approach_contains_ball_contact',
         'approach_pelvis_net_change_deg','approach_torso_net_change_deg']
with pd.option_context('display.max_columns',None,'display.max_rows',None,'display.precision',3):
    display(focus_audit[columns])

fig,axes=plt.subplots(len(focus),2,figsize=(13,max(5,2.25*len(focus))),squeeze=False)
for axs,(_,trial) in zip(axes,focus_audit.iterrows()):
    g=groups[trial.session_swing]
    fc=trial.fp_10_time
    t=g.time.to_numpy(float)
    lo=max(t[0],fc-LOOKBACK_S-.10)
    hi=min(t[-1],max(fc,trial.contact_time if np.isfinite(trial.contact_time) else fc)+.08)
    use=(t>=lo)&(t<=hi)
    for ax,segment in zip(axs,SEGMENTS):
        angle=np.rad2deg(np.unwrap(np.deg2rad(g[segment+'_angle_x'].to_numpy(float))))
        baseline=np.interp(fc-LOOKBACK_S,t,angle)
        ax.plot(1000*(t[use]-fc),angle[use]-baseline,color=COLORS[segment],lw=1.8)
        ax.axvspan(-LOOKBACK_S*1000,0,color='#B3C3CC',alpha=.2,label='Approach window')
        ax.axvline(0,color='black',lw=1,label='Foot contact')
        if np.isfinite(trial.contact_time):
            ax.axvline(1000*(trial.contact_time-fc),color='#D17A22',ls='--',label='Ball contact')
        if np.isfinite(trial.fp_100_time):
            ax.axvline(1000*(trial.fp_100_time-fc),color='#617A56',ls=':',label='Full plant')
        ax.set(title=f'{trial.session_swing} / {segment}',ylabel='Angle change from window start (°)')
        if np.isfinite(trial.contact_time) and trial.contact_time<fc:
            ax.set_title(f'{trial.session_swing} / {segment} / ball contact precedes foot contact',color='#A83C51',fontsize=10)
axes[0,0].legend(fontsize=7,loc='upper left')
for ax in axes[-1]:
    ax.set_xlabel('Milliseconds relative to annotated foot contact')
fig.tight_layout()
fig.savefig(AUDIT_FIG/'focus_trial_event_timelines.png',dpi=160,bbox_inches='tight')
plt.show()


In [ ]:
focus_profile=profiles.loc[profiles.hitter_id.eq(FOCUS_HITTER)&profiles.session_id.eq(FOCUS_SESSION)].iloc[0]
leave_trial_rows=[]
for key in focus.loc[focus.approach_valid,'session_swing']:
    trial_subset=features.loc[features.hitter_id.eq(FOCUS_HITTER)&features.session_swing.ne(key)]
    recalculated=session_profiles(trial_subset,curves)
    row={'omitted_trial':key}
    matching=recalculated.loc[recalculated.session_id.eq(FOCUS_SESSION)] if len(recalculated) else pd.DataFrame()
    for phase in ('approach','landing'):
        row[phase+'_remaining_swings']=int(trial_subset.loc[trial_subset.session_id.eq(FOCUS_SESSION),phase+'_valid'].sum())
        for segment in SEGMENTS:
            metric=f'{phase}_{segment}_relative_trajectory_sd_deg'
            value=matching[metric].iloc[0] if len(matching) else np.nan
            row[metric]=value
            row[metric+'_reduction']=focus_profile[metric]-value
    leave_trial_rows.append(row)
leave_trial_out=pd.DataFrame(leave_trial_rows)
reduction_columns=[f'approach_{s}_relative_trajectory_sd_deg_reduction' for s in SEGMENTS]
leave_trial_out['combined_approach_reduction_deg']=leave_trial_out[reduction_columns].sum(axis=1,min_count=2)
leave_trial_out=leave_trial_out.sort_values('combined_approach_reduction_deg',ascending=False)
leave_trial_out.to_csv(AUDIT_OUT/'focus_leave_one_trial_out.csv',index=False)
DOMINANT_TRIAL=leave_trial_out.omitted_trial.iloc[0]
print('Trial with largest combined reduction in pelvis/torso approach trajectory variability:',DOMINANT_TRIAL)
print('This data-selected omission is an influence check, not a new primary exclusion rule.')
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(leave_trial_out[['omitted_trial','approach_remaining_swings',
        'approach_pelvis_relative_trajectory_sd_deg','approach_torso_relative_trajectory_sd_deg',
        'combined_approach_reduction_deg']])

scenario_features={'full_sample':features,
                   'omit_focus_trial':features.loc[features.session_swing.ne(DOMINANT_TRIAL)].copy(),
                   'omit_focus_hitter':features.loc[features.hitter_id.ne(FOCUS_HITTER)].copy()}
scenario_profiles={'full_sample':profiles}
for label,data in scenario_features.items():
    if label!='full_sample':
        scenario_profiles[label]=session_profiles(data,curves)
    scenario_profiles[label].to_csv(AUDIT_OUT/(label+'_session_profiles.csv'),index=False)
comparison_rows=[]
for label,p in scenario_profiles.items():
    data=scenario_features[label]
    for phase in ('approach','landing'):
        pp=p.loc[p[phase+'_eligible']]
        for segment in SEGMENTS:
            for suffix in ['absolute_trajectory_sd_deg','relative_trajectory_sd_deg','reversal_frequency_pct']:
                metric=f'{phase}_{segment}_{suffix}'
                hitter_values=pp.groupby('hitter_id')[metric].mean().dropna()
                mean,lo,hi=mean_bootstrap(hitter_values)
                comparison_rows.append({'scenario':label,'phase':phase,'segment':segment,'metric':metric,
                    'measure':suffix,'swings':int(data[phase+'_valid'].sum()),'sessions':len(pp),
                    'hitters':len(hitter_values),'mean':mean,'median':hitter_values.median(),
                    'ci_low':lo,'ci_high':hi})
summary_influence=pd.DataFrame(comparison_rows)
reference=summary_influence.loc[summary_influence.scenario.eq('full_sample'),['metric','mean']].rename(columns={'mean':'primary_mean'})
summary_influence=summary_influence.merge(reference,on='metric',validate='many_to_one')
summary_influence['mean_change']=summary_influence['mean']-summary_influence.primary_mean
summary_influence.to_csv(AUDIT_OUT/'cohort_summary_influence.csv',index=False)
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(summary_influence.loc[summary_influence.measure.eq('relative_trajectory_sd_deg'),
        ['scenario','phase','segment','swings','hitters','mean','median','ci_low','ci_high','mean_change']])

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
focus_ids=list(focus.loc[focus.approach_valid,'session_swing'])
for ax,segment in zip(axes,SEGMENTS):
    for key in focus_ids:
        angle=curves[key][('approach',segment)]
        dominant=key==DOMINANT_TRIAL
        ax.plot((PROGRESS-1)*LOOKBACK_S*1000,angle-angle[0],
                color='#D17A22' if dominant else COLORS[segment],alpha=1 if dominant else .35,
                lw=2.5 if dominant else 1,label=key if dominant else None)
    context=focus_audit.loc[focus_audit.session_swing.eq(DOMINANT_TRIAL)].iloc[0]
    if np.isfinite(context.contact_time) and np.isfinite(context.fp_10_time):
        ax.axvline(1000*(context.contact_time-context.fp_10_time),color='black',ls='--',lw=1,label='Annotated ball contact')
    ax.set(title=segment.title()+' / highlighted trial: '+DOMINANT_TRIAL,
           xlabel='Milliseconds before annotated foot contact',ylabel='Angle change from start (°)')
    ax.legend()
fig.tight_layout()
fig.savefig(AUDIT_FIG/'focus_dominant_trial.png',dpi=160,bbox_inches='tight')
plt.show()


In [ ]:
main_metrics=[f'approach_{s}_{kind}_trajectory_sd_deg' for s in SEGMENTS for kind in ('absolute','relative')]
association_rows=[]
for label,p in scenario_profiles.items():
    for outcome in OUTCOMES:
        target=outcome+'_mean'
        for metric in main_metrics:
            if label=='full_sample':
                primary=relationships.loc[relationships.scope.eq('between_hitter') &
                                          relationships.metric.eq(metric) & relationships.outcome.eq(target)]
                if len(primary)!=1:
                    raise ValueError('Expected one primary association for '+metric+' / '+target)
                result=primary.iloc[0].to_dict()
            else:
                paired=p[['hitter_id',metric,target]].dropna()
                between=paired.groupby('hitter_id')[[metric,target]].mean().reset_index()
                result=association(between,metric,target,'between_hitter')
            association_rows.append({'scenario':label,**result})
association_influence=pd.DataFrame(association_rows)
reference=association_influence.loc[association_influence.scenario.eq('full_sample'),['metric','outcome','rho']].rename(columns={'rho':'primary_rho'})
association_influence=association_influence.merge(reference,on=['metric','outcome'],validate='many_to_one')
association_influence['rho_change']=association_influence.rho-association_influence.primary_rho
association_influence.to_csv(AUDIT_OUT/'main_association_influence.csv',index=False)
print('Exploratory associations: nested omission samples are not independent replications.')
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(association_influence[['scenario','metric','outcome','hitters','rho','ci_low','ci_high','rho_change']])

within_rows=[]
for label,data in scenario_features.items():
    for outcome in OUTCOMES:
        for metric in [f'approach_{s}_{suffix}' for s in SEGMENTS for suffix in ('angle_sd_deg','excess_travel_deg','reversals')]:
            d=data.loc[data.approach_valid,['hitter_id','session_id',metric,outcome]].dropna().copy()
            d=d.loc[d.groupby(['hitter_id','session_id'])[metric].transform('size')>=MIN_VALID_SWINGS].copy()
            for column in [metric,outcome]:
                d[column]-=d.groupby(['hitter_id','session_id'])[column].transform('mean')
            within_rows.append({'scenario':label,'metric':metric,'outcome':outcome,'swings':len(d),
                                'hitters':d.hitter_id.nunique(),'rho':safe_rho(d[metric],d[outcome])})
within_influence=pd.DataFrame(within_rows)
within_influence.to_csv(AUDIT_OUT/'within_session_association_influence.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(14,6),sharey=True)
labels=[m.replace('approach_','').replace('_trajectory_sd_deg','').replace('_',' ') for m in main_metrics]
scenario_colors={'full_sample':'#455567','omit_focus_trial':'#D17A22','omit_focus_hitter':'#167D9A'}
for ax,outcome in zip(axes,OUTCOMES):
    for index,(scenario,color) in enumerate(scenario_colors.items()):
        view=association_influence.loc[association_influence.scenario.eq(scenario)&
                                       association_influence.outcome.eq(outcome+'_mean')].set_index('metric').reindex(main_metrics)
        positions=np.arange(len(main_metrics))+(index-1)*.18
        ax.hlines(positions,view.ci_low,view.ci_high,color=color,lw=1.5)
        ax.scatter(view.rho,positions,color=color,s=25,label=scenario.replace('_',' '),zorder=3)
    ax.axvline(0,color='grey',ls='--',lw=1)
    ax.set_yticks(np.arange(len(main_metrics)),labels)
    ax.set(xlabel='Between-hitter Spearman correlation (95% hitter-bootstrap interval)',
           title=outcome.replace('_',' '))
    ax.invert_yaxis()
axes[0].legend(loc='best',fontsize=8)
fig.tight_layout()
fig.savefig(AUDIT_FIG/'main_association_influence.png',dpi=160,bbox_inches='tight')
plt.show()

selected_context=focus_audit.loc[focus_audit.session_swing.eq(DOMINANT_TRIAL)].iloc[0]
focused_lines=[f'Focused hitter: {FOCUS_HITTER}; session: {FOCUS_SESSION}; influential trial: {DOMINANT_TRIAL}.',
    f'Annotated foot contact minus ball contact: {selected_context.foot_contact_minus_ball_contact_ms:+.1f} ms.',
    'A positive offset places ball contact before annotated foot contact; this can contaminate a pre-contact approach interpretation.',
    'The annotations alone do not identify which event is wrong; no event was corrected.',
    f'Global context audit: {int((event_audit.approach_valid & event_audit.foot_contact_after_ball_contact).sum())} retained approach trials have annotated foot contact after ball contact; these flags are exported for review.',
    'The full sample remains primary. Trial and hitter omissions are post hoc influence checks.']
for label,p in scenario_profiles.items():
    target=p.loc[p.hitter_id.eq(FOCUS_HITTER)&p.session_id.eq(FOCUS_SESSION)]
    if len(target):
        for segment in SEGMENTS:
            focused_lines.append(f'{label}: focus {segment} approach trajectory SD = '+
                                 f"{target[f'approach_{segment}_relative_trajectory_sd_deg'].iloc[0]:.3f} degrees.")
focused_lines.append('Small associations and multiple exploratory comparisons do not establish a causal mechanical prescription.')
(AUDIT_OUT/'focused_interpretation.txt').write_text('\n'.join(focused_lines),encoding='utf-8')
print('\n'.join(focused_lines))
focused_manifest={'hitter':FOCUS_HITTER,'session':FOCUS_SESSION,'dominant_trial':DOMINANT_TRIAL,
    'selection_rule':'largest combined reduction in pelvis/torso approach trajectory SD when one focus trial is omitted',
    'primary_sample_preserved':True,'scenarios':list(scenario_profiles),
    'global_approach_foot_contact_after_ball_contact_flags':int((event_audit.approach_valid & event_audit.foot_contact_after_ball_contact).sum()),
    'annotated_foot_contact_minus_ball_contact_ms':float(selected_context.foot_contact_minus_ball_contact_ms),
    'cohort_summary_rows':len(summary_influence),'main_association_rows':len(association_influence),
    'no_event_corrections':True,'no_automatic_exclusions':True}
(AUDIT_OUT/'focused_manifest.json').write_text(json.dumps(focused_manifest,indent=2),encoding='utf-8')


In [ ]:
CHRON_OUT = OUT / 'chronology_sensitivity'
CHRON_FIG = FIG / 'chronology_sensitivity'
for folder in (CHRON_OUT,CHRON_FIG):
    folder.mkdir(parents=True,exist_ok=True)

def classify_approach_chronology(fc,contact,fc_consistent=True,contact_consistent=True):
    if not fc_consistent or not np.isfinite(fc):
        return 'uncheckable_foot_contact'
    if not contact_consistent or not np.isfinite(contact):
        return 'uncheckable_ball_contact'
    return 'foot_contact_after_ball_contact' if fc>contact else 'foot_contact_not_after_ball_contact'

assert classify_approach_chronology(1.,1.2)=='foot_contact_not_after_ball_contact'
assert classify_approach_chronology(1.2,1.)=='foot_contact_after_ball_contact'
assert classify_approach_chronology(1.,np.nan)=='uncheckable_ball_contact'
assert classify_approach_chronology(1.,1.2,contact_consistent=False)=='uncheckable_ball_contact'
assert classify_approach_chronology(np.nan,1.)=='uncheckable_foot_contact'
assert classify_approach_chronology(1.,1.)=='foot_contact_not_after_ball_contact'
chronology_audit=event_audit.copy()
chronology_audit['chronology_status']=[classify_approach_chronology(
    r.fp_10_time,r.contact_time,r.fp_10_time_consistent,r.contact_time_consistent)
    for _,r in chronology_audit.iterrows()]
chronology_audit['excluded_in_sensitivity']=(chronology_audit.approach_valid &
    chronology_audit.chronology_status.eq('foot_contact_after_ball_contact'))
chronology_audit.to_csv(CHRON_OUT/'trial_chronology_classification.csv',index=False)
excluded_chronology_ids=set(chronology_audit.loc[chronology_audit.excluded_in_sensitivity,'session_swing'])
chronology_audit.loc[chronology_audit.excluded_in_sensitivity].to_csv(CHRON_OUT/'excluded_trials.csv',index=False)
chronology_audit.loc[chronology_audit.chronology_status.str.startswith('uncheckable')].to_csv(
    CHRON_OUT/'uncheckable_trials.csv',index=False)
chronology_features=features.loc[~features.session_swing.isin(excluded_chronology_ids)].copy()
chronology_profiles=session_profiles(chronology_features,curves)
chronology_features.to_csv(CHRON_OUT/'screened_swing_metrics.csv',index=False)
chronology_profiles.to_csv(CHRON_OUT/'screened_session_profiles.csv',index=False)
assert set(features.session_swing)-set(chronology_features.session_swing)==excluded_chronology_ids
unknown_ids=set(chronology_audit.loc[chronology_audit.chronology_status.str.startswith('uncheckable'),'session_swing'])
assert unknown_ids.issubset(set(chronology_features.session_swing))
print('Chronology sensitivity: exclude only retained approach trials with annotated foot contact after ball contact.')
print('The same flagged trial IDs are omitted from both phases in this sensitivity; primary results are unchanged.')
print('Missing or inconsistent ball-contact annotations remain uncheckable and are retained, not labeled as confirmed chronology.')
print('Flagged trials removed:',len(excluded_chronology_ids))
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(chronology_audit.loc[chronology_audit.excluded_in_sensitivity,
        ['session_swing','hitter_id','session_id','foot_contact_minus_ball_contact_ms']])

chronology_scenarios={'full_sample':(features,profiles),'exclude_flagged_trials':(chronology_features,chronology_profiles)}
chronology_cohort_rows=[]
chronology_summary_rows=[]
for label,(data,p) in chronology_scenarios.items():
    status=chronology_audit.loc[chronology_audit.session_swing.isin(data.session_swing)]
    for phase in ('approach','landing'):
        valid=data.loc[data[phase+'_valid']]
        eligible=p.loc[p[phase+'_eligible']]
        status_valid=status.loc[status.session_swing.isin(valid.session_swing)]
        chronology_cohort_rows.append({'scenario':label,'phase':phase,'valid_swings':len(valid),
            'hitters_with_valid_swings':valid.hitter_id.nunique(),'eligible_sessions':len(eligible),
            'eligible_hitters':eligible.hitter_id.nunique(),
            'approach_chronology_not_after':int(status_valid.chronology_status.eq('foot_contact_not_after_ball_contact').sum()),
            'approach_chronology_after':int(status_valid.chronology_status.eq('foot_contact_after_ball_contact').sum()),
            'approach_chronology_uncheckable':int(status_valid.chronology_status.str.startswith('uncheckable').sum())})
        for metric in [c for c in eligible if c.startswith((phase+'_pelvis_',phase+'_torso_'))]:
            hitter_values=eligible.groupby('hitter_id')[metric].mean().dropna()
            mean,lo,hi=mean_bootstrap(hitter_values)
            chronology_summary_rows.append({'scenario':label,'phase':phase,'metric':metric,
                'hitters':len(hitter_values),'mean':mean,'median':hitter_values.median(),
                'ci_low':lo,'ci_high':hi})
chronology_cohorts=pd.DataFrame(chronology_cohort_rows)
chronology_cohorts.to_csv(CHRON_OUT/'cohort_comparison.csv',index=False)
chronology_summaries=pd.DataFrame(chronology_summary_rows)
base=chronology_summaries.loc[chronology_summaries.scenario.eq('full_sample'),['metric','mean']].rename(columns={'mean':'primary_mean'})
chronology_summaries=chronology_summaries.merge(base,on='metric',validate='many_to_one')
chronology_summaries['mean_change']=chronology_summaries['mean']-chronology_summaries.primary_mean
chronology_summaries.to_csv(CHRON_OUT/'summary_comparison.csv',index=False)
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(chronology_cohorts)
    display(chronology_summaries.loc[chronology_summaries.metric.str.contains('relative_trajectory_sd_deg'),
        ['scenario','phase','metric','hitters','mean','median','ci_low','ci_high','mean_change']])


In [ ]:
chronology_association_rows=[]
for outcome in OUTCOMES:
    target=outcome+'_mean'
    for metric in main_metrics:
        primary=relationships.loc[relationships.scope.eq('between_hitter') &
                                  relationships.metric.eq(metric) & relationships.outcome.eq(target)]
        if len(primary)!=1:
            raise ValueError('Expected one full-sample association for '+metric+' / '+target)
        chronology_association_rows.append({'scenario':'full_sample',**primary.iloc[0].to_dict()})
        complete=chronology_profiles[['hitter_id',metric,target]].dropna()
        between=complete.groupby('hitter_id')[[metric,target]].mean().reset_index()
        result=association(between,metric,target,'between_hitter')
        chronology_association_rows.append({'scenario':'exclude_flagged_trials',**result})
chronology_associations=pd.DataFrame(chronology_association_rows)
base=chronology_associations.loc[chronology_associations.scenario.eq('full_sample'),['metric','outcome','rho']].rename(columns={'rho':'primary_rho'})
chronology_associations=chronology_associations.merge(base,on=['metric','outcome'],validate='many_to_one')
chronology_associations['rho_change']=chronology_associations.rho-chronology_associations.primary_rho
chronology_associations.to_csv(CHRON_OUT/'main_association_comparison.csv',index=False)
print('Between-hitter correlations and hitter-bootstrap intervals; omission samples are nested, not independent replications.')
with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.precision',3):
    display(chronology_associations[['scenario','metric','outcome','hitters','rho','ci_low','ci_high','rho_change']])

chronology_within_rows=[]
for label,(data,p) in chronology_scenarios.items():
    for outcome in OUTCOMES:
        for metric in [f'approach_{s}_{suffix}' for s in SEGMENTS for suffix in ('angle_sd_deg','excess_travel_deg','reversals')]:
            d=data.loc[data.approach_valid,['hitter_id','session_id',metric,outcome]].dropna().copy()
            d=d.loc[d.groupby(['hitter_id','session_id'])[metric].transform('size')>=MIN_VALID_SWINGS].copy()
            for column in [metric,outcome]:
                d[column]-=d.groupby(['hitter_id','session_id'])[column].transform('mean')
            chronology_within_rows.append({'scenario':label,'metric':metric,'outcome':outcome,
                'swings':len(d),'hitters':d.hitter_id.nunique(),'rho':safe_rho(d[metric],d[outcome])})
chronology_within=pd.DataFrame(chronology_within_rows)
chronology_within.to_csv(CHRON_OUT/'within_session_association_comparison.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(14,6),sharey=True)
labels=[m.replace('approach_','').replace('_trajectory_sd_deg','').replace('_',' ') for m in main_metrics]
for ax,outcome in zip(axes,OUTCOMES):
    for index,(scenario,color) in enumerate([('full_sample','#455567'),('exclude_flagged_trials','#167D9A')]):
        view=chronology_associations.loc[chronology_associations.scenario.eq(scenario)&
            chronology_associations.outcome.eq(outcome+'_mean')].set_index('metric').reindex(main_metrics)
        positions=np.arange(len(main_metrics))+(index-.5)*.20
        ax.hlines(positions,view.ci_low,view.ci_high,color=color,lw=1.6)
        ax.scatter(view.rho,positions,color=color,s=28,label=scenario.replace('_',' '),zorder=3)
    ax.axvline(0,color='grey',ls='--',lw=1)
    ax.set_yticks(np.arange(len(main_metrics)),labels)
    ax.set(xlabel='Between-hitter Spearman correlation (95% hitter-bootstrap interval)',
           title=outcome.replace('_',' '))
axes[0].invert_yaxis()
axes[0].legend(loc='best',fontsize=8)
fig.tight_layout()
fig.savefig(CHRON_FIG/'main_association_comparison.png',dpi=160,bbox_inches='tight')
plt.show()

screened=chronology_associations.loc[chronology_associations.scenario.eq('exclude_flagged_trials')]
maximum_change=float(screened.rho_change.abs().max())
chronology_lines=[f'Chronology sensitivity removed {len(excluded_chronology_ids)} trials with annotated foot contact after ball contact.',
    'The primary full sample is unchanged; the sensitivity omits flagged trial IDs from both phase analyses.',
    f'{len(unknown_ids)} retained trial IDs have uncheckable chronology; they remain labeled uncheckable in the sensitivity.',
    'This rule identifies an annotation/window interpretation concern and does not repair event times or establish which event is incorrect.',
    f'Maximum absolute change across the eight main between-hitter correlations: {maximum_change:.3f}.',
    'Session eligibility and outcome profiles were recomputed after trial removal.',
    'Intervals are exploratory; agreement under this screen is not causal evidence or external validation.']
for phase in ('approach','landing'):
    row=chronology_cohorts.loc[chronology_cohorts.scenario.eq('exclude_flagged_trials')&chronology_cohorts.phase.eq(phase)].iloc[0]
    chronology_lines.append(f'{phase.title()}: {row.valid_swings} valid swings, {row.eligible_sessions} eligible sessions.')
(CHRON_OUT/'interpretation.txt').write_text('\n'.join(chronology_lines),encoding='utf-8')
chronology_manifest={'excluded_trial_ids':sorted(excluded_chronology_ids),
    'excluded_count':len(excluded_chronology_ids),'uncheckable_trial_ids':sorted(unknown_ids),
    'unknowns_retained_and_labeled':True,'full_sample_preserved':True,
    'rule':'omit retained approach trials where consistent finite fp_10_time > consistent finite contact_time',
    'scope':'same flagged trial IDs omitted from both phases; session/outcome profiles recomputed',
    'cohort_comparison':chronology_cohorts.to_dict(orient='records'),
    'main_associations':len(main_metrics)*len(OUTCOMES),
    'maximum_absolute_rho_change':maximum_change,'bootstraps':BOOTSTRAPS,'seed':SEED}
(CHRON_OUT/'manifest.json').write_text(json.dumps(chronology_manifest,indent=2),encoding='utf-8')
print('\n'.join(chronology_lines))


In [ ]:
trajectory_rows=[]
for key,trace in curves.items():
    for (phase,s),a in trace.items():
        trajectory_rows.extend({'session_swing':key,'phase':phase,'segment':s,
                                'progress':float(p),'angle_deg':float(value),
                                'start_relative_angle_deg':float(value-a[0])}
                               for p,value in zip(PROGRESS,a))
pd.DataFrame(trajectory_rows).to_csv(OUT/'normalized_trajectories.csv',index=False)
manifest={'study':'stride_movement_consistency','dataset':'OpenBiomechanics Dataset v1',
          'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
          'settings':settings,'primary_minimum_swings':MIN_VALID_SWINGS,
          'filter_padding_s':FILTER_PADDING_S,'progress_points':len(PROGRESS),
          'bootstraps':BOOTSTRAPS,'seed':SEED,'retained_swings':len(features),
          'retained_hitters':features.hitter_id.nunique(),'eligible_sessions':len(profiles),
          'phase_cohorts':cohorts.to_dict(orient='records'),
          'phase_eligibility':'independent; approach requires foot contact, landing requires contact and plant; ball contact is unused',
          'reversal_frequency':'percentage of phase-valid session swings with at least one threshold-confirmed reversal',
          'input_sha256':{name:file_hash(RAW/name) for name in
                          ['joint_angles.csv','metadata.csv','poi_metrics.csv'] if (RAW/name).exists()},
          'archive_expected_sha256':ARCHIVE_HASHES,
          'optional_outcomes_observed':{o:int(features[o].notna().sum()) for o in OUTCOMES},
          'measurement_checks':'passed during this run',
          'scope':'fixed pre-contact window, not measured full stride; descriptive observational analysis',
          'focused_outlier_check':focused_manifest,'chronology_sensitivity':chronology_manifest}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
lines=['Stride Movement Consistency — descriptive run',
       f'Retained {len(features)} swings from {features.hitter_id.nunique()} hitters.',
       f'{len(profiles)} sessions met the {MIN_VALID_SWINGS}-swing minimum in at least one phase.',
       'Primary window: 150 ms before front-foot contact; landing analyzed separately.',
       'Trajectory variability measures within-session repeatability, not longitudinal reliability.']
for phase in ('approach','landing'):
  for s in SEGMENTS:
    metric=f'{phase}_{s}_relative_trajectory_sd_deg'
    row=summary.loc[summary.metric.eq(metric)].iloc[0]
    lines.append(f"{phase.title()} {s} mean start-relative trajectory SD: {row['mean']:.2f} degrees "
                 f"(hitter-bootstrap 95% interval {row.ci_low:.2f} to {row.ci_high:.2f}).")
lines.extend(['Check window/filter/minimum-swing sensitivity before interpreting rankings.',
              'No universal ideal, movement-quality score, or causal performance benefit is established.',
              'Individual curves and derived trial CSVs are local outputs; review the OBP data license before redistribution.'])
(OUT/'interpretation.txt').write_text('\n'.join(lines),encoding='utf-8')
print('\n'.join(lines))
print('Results:',OUT)
print('Figures:',FIG)
print('Chronology sensitivity summary:')
print('\n'.join(chronology_lines))
